In [50]:
# Imports
import pandas as pd
from mssql_python import connect
import config

In [51]:
# Paths
films_clean_path = "../transformation/transformed_data/films_clean_data.csv"
oscars__clean_path = "../transformation/transformed_data/oscars_clean_data.csv"
streaming_clean_path = "../transformation/transformed_data/streaming_clean_data.csv"

films_clean_df = pd.read_csv(films_clean_path)
oscars__clean_df = pd.read_csv(films_clean_path)
streaming_clean_df = pd.read_csv(films_clean_path)

## Setting up connection to Azure SQL Database

In [52]:
connection_string = (
    f"Server=tcp:{config.SERVER};"
    f"Database={config.DB_NAME};"
    f"UID={config.USERNAME};"
    f"PWD={config.PASSWORD};"
    "Encrypt=yes;TrustServerCertificate=no;"
)

def get_conn():
    conn = connect(connection_string)
    conn.setautocommit(True)
    return conn

get_conn()

In [53]:
import pandas as pd
from mssql_python import connect
import config

films_clean_path = "../transformation/transformed_data/films_clean_data.csv"
oscars_clean_path = "../transformation/transformed_data/oscars_clean_data.csv"
streaming_clean_path = "../transformation/transformed_data/streaming_clean_data.csv"

films_df = pd.read_csv(films_clean_path)
oscars_df = pd.read_csv(oscars_clean_path)
streaming_df = pd.read_csv(streaming_clean_path)

print("Loaded CSVs:")
print("  films_df     :", films_df.shape)
print("  oscars_df    :", oscars_df.shape)
print("  streaming_df :", streaming_df.shape)

print("\nColumns preview:")
print("films_df columns:", list(films_df.columns))
print("oscars_df columns:", list(oscars_df.columns))
print("streaming_df columns:", list(streaming_df.columns))

Loaded CSVs:
  films_df     : (103, 9)
  oscars_df    : (240, 14)
  streaming_df : (92, 8)

Columns preview:
films_df columns: ['title', 'year', 'rating', 'genres', 'runtime', 'director', 'imdb_id', 'rating_out_of_range', 'runtime_suspicious']
oscars_df columns: ['ceremony', 'year', 'class', 'canonicalcategory', 'category', 'nomid', 'film', 'filmid', 'name', 'nominees', 'nomineeids', 'winner', 'detail', 'yearstart']
streaming_df columns: ['name', 'release_year', 'poster', 'imdbid', 'streaming', 'rent', 'buy', 'num_streaming']


In [54]:
# Films columns
film_title_col = "title"
film_year_col  = "year"
genre_col      = "genres"

# Streaming columns (NOTE: service values are spread across multiple columns)
stream_title_col = "name"
stream_year_col  = "release_year"
stream_service_cols = ["streaming", "rent", "buy"]  # these contain service names/lists

# Oscars columns
oscar_year_col  = "year"
oscar_cat_col   = "category"
oscar_nom_col   = "name"
oscar_win_col   = "winner"
oscar_title_col = "film"

print("✅ Column mappings locked in.")

✅ Column mappings locked in.


In [55]:
import pandas as pd

def split_services(x):
    """Split service strings into a list. Handles NaN, lists-ish strings, comma/pipe separators."""
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s or s.lower() in ("none", "nan", "null", "[]"):
        return []
    # normalize common separators
    s = s.replace("|", ",").replace(";", ",")
    # remove brackets/quotes if present
    s = s.replace("[", "").replace("]", "").replace('"', "").replace("'", "")
    parts = [p.strip() for p in s.split(",")]
    return [p for p in parts if p]

links = []
for _, r in streaming_df.iterrows():
    title = str(r[stream_title_col]).strip() if pd.notna(r[stream_title_col]) else None
    year  = int(r[stream_year_col]) if pd.notna(r[stream_year_col]) else None
    if not title:
        continue

    for access_type in stream_service_cols:
        for svc in split_services(r.get(access_type)):
            links.append((title, year, svc, access_type))

stream_links_df = pd.DataFrame(
    links, columns=["film_title", "film_year", "service_name", "access_type"]
)

print("✅ stream_links_df built:", stream_links_df.shape)
print("Unique services:", stream_links_df["service_name"].nunique())
print("\nTop 10 services:")
print(stream_links_df["service_name"].value_counts().head(10))
print("\nPreview rows:")
display(stream_links_df.head(15))

✅ stream_links_df built: (605, 4)
Unique services: 124

Top 10 services:
service_name
Fandango At Home ($3.99)        39
Amazon Video ($3.99)            37
Apple TV Store ($3.99)          35
Google Play Movies ($3.99)      34
YouTube ($3.99)                 34
Spectrum On Demand ($3.99)      21
Amazon DVD / Blu-ray ($None)    21
Netflix                         20
Netflix Standard with Ads       19
Disney Plus                     11
Name: count, dtype: int64

Preview rows:


,film_title,film_year,service_name,access_type
0,Maestro,2023,Netflix,streaming
1,Maestro,2023,Netflix Standard with Ads,streaming
2,Rustin,2023,Netflix,streaming
3,Rustin,2023,Netflix Standard with Ads,streaming
4,The Holdovers,2023,Amazon Video ($3.99),rent
5,The Holdovers,2023,Apple TV Store ($3.99),rent
6,The Holdovers,2023,Google Play Movies ($3.99),rent
7,The Holdovers,2023,YouTube ($3.99),rent
8,The Holdovers,2023,Fandango At Home ($3.99),rent
9,The Holdovers,2023,FlixFling ($3.99),rent


In [56]:
# Reconnect (fresh connection)
conn = get_conn()
cursor = conn.cursor()

# Ping
cursor.execute("SELECT 1;")
print("✅ Connection alive:", cursor.fetchone()[0])

✅ Connection alive: 1


In [57]:
# 1) film_genre
cursor.execute("""
IF OBJECT_ID('dbo.film_genre','U') IS NULL
CREATE TABLE dbo.film_genre(
    film_genre_id INT IDENTITY(1,1) PRIMARY KEY,
    genre_name VARCHAR(100) NOT NULL UNIQUE
);
""")
print("✅ film_genre ensured")

# 2) film
cursor.execute("""
IF OBJECT_ID('dbo.film','U') IS NULL
CREATE TABLE dbo.film(
    film_id INT IDENTITY(1,1) PRIMARY KEY,
    film_title VARCHAR(300) NOT NULL,
    film_year INT NULL,
    film_genre_id INT NULL,
    CONSTRAINT FK_film_genre FOREIGN KEY (film_genre_id) REFERENCES dbo.film_genre(film_genre_id),
    CONSTRAINT UQ_film_title_year UNIQUE (film_title, film_year)
);
""")
print("✅ film ensured")

# 3) streaming_service
cursor.execute("""
IF OBJECT_ID('dbo.streaming_service','U') IS NULL
CREATE TABLE dbo.streaming_service(
    streaming_service_id INT IDENTITY(1,1) PRIMARY KEY,
    service_name VARCHAR(200) NOT NULL UNIQUE
);
""")
print("✅ streaming_service ensured")

# 4) film_streaming
cursor.execute("""
IF OBJECT_ID('dbo.film_streaming','U') IS NULL
CREATE TABLE dbo.film_streaming(
    film_streaming_id INT IDENTITY(1,1) PRIMARY KEY,
    film_id INT NOT NULL,
    streaming_service_id INT NOT NULL,
    access_type VARCHAR(20) NULL,
    CONSTRAINT FK_film_streaming_film FOREIGN KEY (film_id) REFERENCES dbo.film(film_id),
    CONSTRAINT FK_film_streaming_service FOREIGN KEY (streaming_service_id) REFERENCES dbo.streaming_service(streaming_service_id),
    CONSTRAINT UQ_film_service_access UNIQUE (film_id, streaming_service_id, access_type)
);
""")
print("✅ film_streaming ensured")

# 5) oscar_award
cursor.execute("""
IF OBJECT_ID('dbo.oscar_award','U') IS NULL
CREATE TABLE dbo.oscar_award(
    oscar_award_id INT IDENTITY(1,1) PRIMARY KEY,
    film_id INT NULL,
    award_year INT NULL,
    category VARCHAR(300) NULL,
    nominee VARCHAR(400) NULL,
    winner BIT NULL,
    raw_film_title VARCHAR(300) NULL,
    CONSTRAINT FK_oscar_film FOREIGN KEY (film_id) REFERENCES dbo.film(film_id)
);
""")
print("✅ oscar_award ensured")

✅ film_genre ensured
✅ film ensured
✅ streaming_service ensured
✅ film_streaming ensured
✅ oscar_award ensured


In [58]:
# Force Unicode on the way out
cursor.execute("""
SELECT
  film_id,
  CAST(film_title AS NVARCHAR(300)) AS film_title,
  film_year
FROM dbo.film;
""")

rows = cursor.fetchall()
film_key_to_id = {(t, y): fid for (fid, t, y) in rows}

print("✅ Films in DB (map rebuilt):", len(film_key_to_id))

cursor.execute("""
SELECT TOP 5
  film_id,
  CAST(film_title AS NVARCHAR(300)) AS film_title,
  film_year
FROM dbo.film
ORDER BY film_id DESC;
""")
print("Sample films:", cursor.fetchall())

✅ Films in DB (map rebuilt): 83
Sample films: [(83, 'September 5', 2024), (82, 'Kingdom of the Planet of the Apes', 2024), (81, 'Better Man', 2024), (80, 'Alien: Romulus', 2024), (79, 'Anuja', 2024)]


In [59]:
# Reconnect safe
conn = get_conn()
cursor = conn.cursor()
cursor.execute("SELECT 1;")
print("✅ Connection alive:", cursor.fetchone()[0])

# Add column if missing
cursor.execute("""
IF COL_LENGTH('dbo.film_streaming', 'access_type') IS NULL
BEGIN
    ALTER TABLE dbo.film_streaming ADD access_type VARCHAR(20) NULL;
END
""")
print("✅ access_type column ensured")

✅ Connection alive: 1
✅ access_type column ensured


In [60]:
# Drop old unique constraint if it exists (common name from your earlier code)
cursor.execute("""
IF EXISTS (
    SELECT 1 FROM sys.key_constraints
    WHERE name = 'UQ_film_service' AND parent_object_id = OBJECT_ID('dbo.film_streaming')
)
BEGIN
    ALTER TABLE dbo.film_streaming DROP CONSTRAINT UQ_film_service;
END
""")
print("✅ Old unique constraint dropped (if it existed)")

# Add new unique constraint that includes access_type
cursor.execute("""
IF NOT EXISTS (
    SELECT 1 FROM sys.key_constraints
    WHERE name = 'UQ_film_service_access' AND parent_object_id = OBJECT_ID('dbo.film_streaming')
)
BEGIN
    ALTER TABLE dbo.film_streaming
    ADD CONSTRAINT UQ_film_service_access UNIQUE (film_id, streaming_service_id, access_type);
END
""")
print("✅ New unique constraint ensured")

✅ Old unique constraint dropped (if it existed)
✅ New unique constraint ensured


In [61]:
import pandas as pd

def norm_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None

def norm_int(x):
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except:
        return None

# Rebuild svc_map safely (Unicode)
cursor.execute("""
SELECT streaming_service_id, CAST(service_name AS NVARCHAR(200)) AS service_name
FROM dbo.streaming_service;
""")
svc_map = {name: sid for (sid, name) in cursor.fetchall()}

inserted = 0
skipped_no_film = 0
skipped_no_service = 0

for _, r in stream_links_df.iterrows():
    t = norm_str(r["film_title"])
    y = norm_int(r["film_year"])
    s = norm_str(r["service_name"])
    a = norm_str(r["access_type"])

    film_id = film_key_to_id.get((t, y)) or film_key_to_id.get((t, None))
    svc_id = svc_map.get(s)

    if not svc_id:
        skipped_no_service += 1
        continue
    if not film_id:
        skipped_no_film += 1
        continue

    cursor.execute(
        """
        IF NOT EXISTS (
            SELECT 1 FROM dbo.film_streaming
            WHERE film_id = ? AND streaming_service_id = ? AND access_type = ?
        )
        INSERT INTO dbo.film_streaming (film_id, streaming_service_id, access_type)
        VALUES (?, ?, ?);
        """,
        (film_id, svc_id, a, film_id, svc_id, a)
    )
    inserted += 1

print("✅ Links attempted:", inserted)
print("⚠️ Skipped (no matching film):", skipped_no_film)
print("⚠️ Skipped (no matching service):", skipped_no_service)

cursor.execute("SELECT COUNT(*) FROM dbo.film_streaming;")
print("dbo.film_streaming count:", cursor.fetchone()[0])

cursor.execute("""
SELECT TOP 8
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year,
  CAST(s.service_name AS NVARCHAR(200)) AS service_name,
  fs.access_type
FROM dbo.film_streaming fs
JOIN dbo.film f ON fs.film_id = f.film_id
JOIN dbo.streaming_service s ON fs.streaming_service_id = s.streaming_service_id
ORDER BY fs.film_streaming_id DESC;
""")
print("Sample links:", cursor.fetchall())

✅ Links attempted: 0
⚠️ Skipped (no matching film): 0
⚠️ Skipped (no matching service): 605
dbo.film_streaming count: 0
Sample links: []


In [62]:
# ---- Counts
cursor.execute("SELECT COUNT(*) FROM dbo.film;")
film_ct = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM dbo.film_genre;")
genre_ct = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM dbo.streaming_service;")
svc_ct = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM dbo.film_streaming;")
link_ct = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM dbo.oscar_award;")
osc_ct = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM dbo.oscar_award WHERE film_id IS NOT NULL;")
osc_linked_ct = cursor.fetchone()[0]

print("✅ ROW COUNTS")
print("dbo.film            :", film_ct)
print("dbo.film_genre      :", genre_ct)
print("dbo.streaming_service:", svc_ct)
print("dbo.film_streaming  :", link_ct)
print("dbo.oscar_award     :", osc_ct)
print("dbo.oscar_award linked to film:", osc_linked_ct)

# ---- Join check: film + genre
cursor.execute("""
SELECT TOP 10
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year,
  CAST(g.genre_name AS NVARCHAR(100)) AS genre
FROM dbo.film f
LEFT JOIN dbo.film_genre g ON f.film_genre_id = g.film_genre_id
ORDER BY f.film_id DESC;
""")
print("\n✅ Sample film + genre:", cursor.fetchall())

# ---- Join check: film + streaming
cursor.execute("""
SELECT TOP 10
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year,
  CAST(s.service_name AS NVARCHAR(200)) AS service_name,
  fs.access_type
FROM dbo.film_streaming fs
JOIN dbo.film f ON fs.film_id = f.film_id
JOIN dbo.streaming_service s ON fs.streaming_service_id = s.streaming_service_id
ORDER BY fs.film_streaming_id DESC;
""")
print("\n✅ Sample film + streaming:", cursor.fetchall())

# ---- Join check: oscar + film (only linked)
cursor.execute("""
SELECT TOP 10
  oa.award_year,
  CAST(oa.category AS NVARCHAR(300)) AS category,
  CAST(oa.raw_film_title AS NVARCHAR(300)) AS oscar_film_title,
  CAST(f.film_title AS NVARCHAR(300)) AS linked_film_title,
  oa.winner
FROM dbo.oscar_award oa
JOIN dbo.film f ON oa.film_id = f.film_id
ORDER BY oa.oscar_award_id DESC;
""")
print("\n✅ Sample oscar + linked film:", cursor.fetchall())

✅ ROW COUNTS
dbo.film            : 83
dbo.film_genre      : 0
dbo.streaming_service: 0
dbo.film_streaming  : 0
dbo.oscar_award     : 0
dbo.oscar_award linked to film: 0

✅ Sample film + genre: [('September 5', 2024, None), ('Kingdom of the Planet of the Apes', 2024, None), ('Better Man', 2024, None), ('Alien: Romulus', 2024, None), ('Anuja', 2024, None), ('Nickel Boys', 2024, None), ('Elton John: Never Too Late', 2024, None), ('The Six Triple Eight', 2024, None), ('A Different Man', 2024, None), ("Dane-ye anjir-e ma'abed", 2024, None)]

✅ Sample film + streaming: []

✅ Sample oscar + linked film: []


In [63]:
def insert_missing_films_from_streaming(cursor, stream_links_df):
    inserted_missing = 0

    # Load existing films into a lookup map
    cursor.execute("""
        SELECT film_id, CAST(film_title AS NVARCHAR(300)) AS film_title, film_year
        FROM dbo.film;
    """)
    film_key_to_id = {(t, y): fid for (fid, t, y) in cursor.fetchall()}

    # Iterate through unique streaming films
    for _, r in stream_links_df[["film_title", "film_year"]].drop_duplicates().iterrows():
        t = norm_str(r["film_title"])
        y = norm_int(r["film_year"])

        if not t:
            continue

        # Skip if already in DB
        if (t, y) in film_key_to_id:
            continue

        # Insert if not exists (SQL-level safety)
        cursor.execute(
            """
            IF NOT EXISTS (
                SELECT 1 FROM dbo.film
                WHERE film_title = ?
                  AND ((film_year IS NULL AND ? IS NULL) OR film_year = ?)
            )
            INSERT INTO dbo.film (film_title, film_year, film_genre_id)
            VALUES (?, ?, NULL);
            """,
            (t, y, y, t, y)
        )

        inserted_missing += 1

    print("✅ Missing films inserted from streaming:", inserted_missing)
    return inserted_missing

In [64]:
cursor.execute("""
CREATE OR ALTER VIEW dbo.vw_film_summary AS
SELECT
  f.film_id,
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year,
  CAST(g.genre_name AS NVARCHAR(100)) AS genre,
  STRING_AGG(
    CONCAT(CAST(s.service_name AS NVARCHAR(200)), ' [', fs.access_type, ']'),
    '; '
  ) WITHIN GROUP (ORDER BY s.service_name) AS availability
FROM dbo.film f
LEFT JOIN dbo.film_genre g ON f.film_genre_id = g.film_genre_id
LEFT JOIN dbo.film_streaming fs ON f.film_id = fs.film_id
LEFT JOIN dbo.streaming_service s ON fs.streaming_service_id = s.streaming_service_id
GROUP BY f.film_id, f.film_title, f.film_year, g.genre_name;
""")

cursor.execute("SELECT TOP 10 * FROM dbo.vw_film_summary ORDER BY film_id DESC;")
print("✅ vw_film_summary sample:", cursor.fetchall())

✅ vw_film_summary sample: [(83, 'September 5', 2024, None, ' []'), (82, 'Kingdom of the Planet of the Apes', 2024, None, ' []'), (81, 'Better Man', 2024, None, ' []'), (80, 'Alien: Romulus', 2024, None, ' []'), (79, 'Anuja', 2024, None, ' []'), (78, 'Nickel Boys', 2024, None, ' []'), (77, 'Elton John: Never Too Late', 2024, None, ' []'), (76, 'The Six Triple Eight', 2024, None, ' []'), (75, 'A Different Man', 2024, None, ' []'), (74, "Dane-ye anjir-e ma'abed", 2024, None, ' []')]


In [65]:
# Refresh film map (unicode-safe)
cursor.execute("""
SELECT film_id, CAST(film_title AS NVARCHAR(300)) AS film_title, film_year
FROM dbo.film;
""")
film_key_to_id = {(t, y): fid for (fid, t, y) in cursor.fetchall()}

# Refresh service map (unicode-safe)
cursor.execute("""
SELECT streaming_service_id, CAST(service_name AS NVARCHAR(200)) AS service_name
FROM dbo.streaming_service;
""")
svc_map = {name: sid for (sid, name) in cursor.fetchall()}

inserted = 0
skipped_no_film = 0

for _, r in stream_links_df.iterrows():
    t = norm_str(r["film_title"])
    y = norm_int(r["film_year"])
    s = norm_str(r["service_name"])
    a = norm_str(r["access_type"])

    film_id = film_key_to_id.get((t, y)) or film_key_to_id.get((t, None))
    svc_id = svc_map.get(s)

    if not film_id:
        skipped_no_film += 1
        continue
    if not svc_id:
        continue

    cursor.execute(
        """
        IF NOT EXISTS (
            SELECT 1 FROM dbo.film_streaming
            WHERE film_id = ? AND streaming_service_id = ? AND access_type = ?
        )
        INSERT INTO dbo.film_streaming (film_id, streaming_service_id, access_type)
        VALUES (?, ?, ?);
        """,
        (film_id, svc_id, a, film_id, svc_id, a)
    )
    inserted += 1

cursor.execute("SELECT COUNT(*) FROM dbo.film_streaming;")
total_links = cursor.fetchone()[0]

print("✅ New links attempted:", inserted)
print("⚠️ Still skipped (no matching film):", skipped_no_film)
print("✅ dbo.film_streaming total:", total_links)

✅ New links attempted: 0
⚠️ Still skipped (no matching film): 1
✅ dbo.film_streaming total: 0


In [66]:
cursor.execute("""
CREATE OR ALTER VIEW dbo.vw_film_summary AS
SELECT
  f.film_id,
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year,
  CAST(g.genre_name AS NVARCHAR(100)) AS genre,
  NULLIF(
    STRING_AGG(
      CONCAT(CAST(s.service_name AS NVARCHAR(200)), ' [', fs.access_type, ']'),
      '; '
    ) WITHIN GROUP (ORDER BY s.service_name),
    ''
  ) AS availability
FROM dbo.film f
LEFT JOIN dbo.film_genre g ON f.film_genre_id = g.film_genre_id
LEFT JOIN dbo.film_streaming fs ON f.film_id = fs.film_id
LEFT JOIN dbo.streaming_service s ON fs.streaming_service_id = s.streaming_service_id
GROUP BY f.film_id, f.film_title, f.film_year, g.genre_name;
""")

cursor.execute("""
SELECT TOP 10 film_id, film_title, film_year, genre, availability
FROM dbo.vw_film_summary
ORDER BY film_id DESC;
""")
print("✅ vw_film_summary sample:", cursor.fetchall())

✅ vw_film_summary sample: [(83, 'September 5', 2024, None, ' []'), (82, 'Kingdom of the Planet of the Apes', 2024, None, ' []'), (81, 'Better Man', 2024, None, ' []'), (80, 'Alien: Romulus', 2024, None, ' []'), (79, 'Anuja', 2024, None, ' []'), (78, 'Nickel Boys', 2024, None, ' []'), (77, 'Elton John: Never Too Late', 2024, None, ' []'), (76, 'The Six Triple Eight', 2024, None, ' []'), (75, 'A Different Man', 2024, None, ' []'), (74, "Dane-ye anjir-e ma'abed", 2024, None, ' []')]


In [67]:
cursor.execute("""
CREATE OR ALTER VIEW dbo.vw_film_summary AS
SELECT
  f.film_id,
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year,
  CAST(g.genre_name AS NVARCHAR(100)) AS genre,
  CASE 
    WHEN COUNT(fs.film_streaming_id) = 0 THEN NULL
    ELSE STRING_AGG(
      CONCAT(CAST(s.service_name AS NVARCHAR(200)), ' [', fs.access_type, ']'),
      '; '
    ) WITHIN GROUP (ORDER BY s.service_name)
  END AS availability
FROM dbo.film f
LEFT JOIN dbo.film_genre g ON f.film_genre_id = g.film_genre_id
LEFT JOIN dbo.film_streaming fs ON f.film_id = fs.film_id
LEFT JOIN dbo.streaming_service s ON fs.streaming_service_id = s.streaming_service_id
GROUP BY f.film_id, f.film_title, f.film_year, g.genre_name;
""")

cursor.execute("""
SELECT TOP 10 film_id, film_title, film_year, availability
FROM dbo.vw_film_summary
ORDER BY film_id DESC;
""")
print(cursor.fetchall())

[(83, 'September 5', 2024, None), (82, 'Kingdom of the Planet of the Apes', 2024, None), (81, 'Better Man', 2024, None), (80, 'Alien: Romulus', 2024, None), (79, 'Anuja', 2024, None), (78, 'Nickel Boys', 2024, None), (77, 'Elton John: Never Too Late', 2024, None), (76, 'The Six Triple Eight', 2024, None), (75, 'A Different Man', 2024, None), (74, "Dane-ye anjir-e ma'abed", 2024, None)]


In [68]:
cursor.execute("""
SELECT COUNT(*) AS no_availability
FROM dbo.vw_film_summary
WHERE availability IS NULL;
""")

print("Films with no availability:", cursor.fetchone()[0])

Films with no availability: 83


In [69]:
cursor.execute("""
SELECT 
  f.film_id,
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year
FROM dbo.film f
LEFT JOIN dbo.film_streaming fs
  ON f.film_id = fs.film_id
WHERE fs.film_id IS NULL
ORDER BY f.film_year DESC, f.film_title;
""")

no_streaming = cursor.fetchall()
print("Films with NO streaming links:", len(no_streaming))
print(no_streaming[:20])

Films with NO streaming links: 83
[(47, 'A Complete Unknown', 2024), (75, 'A Different Man', 2024), (52, 'A Real Pain', 2024), (80, 'Alien: Romulus', 2024), (51, 'Anora', 2024), (79, 'Anuja', 2024), (81, 'Better Man', 2024), (66, 'Burakku bokkusu daiarîzu', 2024), (49, 'Conclave', 2024), (74, "Dane-ye anjir-e ma'abed", 2024), (62, 'Dune: Part Two', 2024), (77, 'Elton John: Never Too Late', 2024), (54, 'EMILIA PÉREZ', 2024), (65, 'Gladiator II', 2024), (71, 'I Am Ready, Warden', 2024), (56, "I'm Still Here", 2024), (58, 'Inside Out 2', 2024), (82, 'Kingdom of the Planet of the Apes', 2024), (36, 'Letter to a Pig', 2024), (63, 'Maria', 2024)]


In [70]:
import unicodedata

def normalize_title(t):
    if not t:
        return None
    t = str(t).strip().lower()
    # remove accents
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    # normalize apostrophes
    t = t.replace("’", "'").replace("`", "'")
    return t

# Rebuild film map with normalized keys
cursor.execute("""
SELECT film_id, CAST(film_title AS NVARCHAR(300)) AS film_title, film_year
FROM dbo.film;
""")

film_key_to_id_norm = {}
for fid, t, y in cursor.fetchall():
    key = (normalize_title(t), y)
    film_key_to_id_norm[key] = fid
    # also allow year-less fallback
    film_key_to_id_norm[(normalize_title(t), None)] = fid

print("✅ Normalized film map built:", len(film_key_to_id_norm))

✅ Normalized film map built: 166


In [71]:
# Refresh service map
cursor.execute("""
SELECT streaming_service_id, CAST(service_name AS NVARCHAR(200)) AS service_name
FROM dbo.streaming_service;
""")
svc_map = {name: sid for (sid, name) in cursor.fetchall()}

inserted = 0
skipped = 0

for _, r in stream_links_df.iterrows():
    t = normalize_title(r["film_title"])
    y = norm_int(r["film_year"])
    s = norm_str(r["service_name"])
    a = norm_str(r["access_type"])

    film_id = film_key_to_id_norm.get((t, y)) or film_key_to_id_norm.get((t, None))
    svc_id = svc_map.get(s)

    if not film_id or not svc_id:
        skipped += 1
        continue

    cursor.execute(
        """
        IF NOT EXISTS (
            SELECT 1 FROM dbo.film_streaming
            WHERE film_id = ? AND streaming_service_id = ? AND access_type = ?
        )
        INSERT INTO dbo.film_streaming (film_id, streaming_service_id, access_type)
        VALUES (?, ?, ?);
        """,
        (film_id, svc_id, a, film_id, svc_id, a)
    )
    inserted += 1

print("✅ Additional links inserted:", inserted)
print("⚠️ Still skipped:", skipped)

cursor.execute("SELECT COUNT(*) FROM dbo.film_streaming;")
print("dbo.film_streaming total:", cursor.fetchone()[0])

✅ Additional links inserted: 0
⚠️ Still skipped: 605
dbo.film_streaming total: 0


In [72]:
cursor.execute("""
SELECT COUNT(*)
FROM dbo.vw_film_summary
WHERE availability IS NULL;
""")
print("Films with no availability:", cursor.fetchone()[0])

Films with no availability: 83


In [73]:
cursor.execute("""
SELECT 
  f.film_id,
  CAST(f.film_title AS NVARCHAR(300)) AS film_title,
  f.film_year
FROM dbo.film f
LEFT JOIN dbo.film_streaming fs ON f.film_id = fs.film_id
WHERE fs.film_id IS NULL
ORDER BY f.film_year DESC, f.film_title;
""")

missing = cursor.fetchall()
print("Films with NO availability:", len(missing))
print(missing)

Films with NO availability: 83
[(47, 'A Complete Unknown', 2024), (75, 'A Different Man', 2024), (52, 'A Real Pain', 2024), (80, 'Alien: Romulus', 2024), (51, 'Anora', 2024), (79, 'Anuja', 2024), (81, 'Better Man', 2024), (66, 'Burakku bokkusu daiarîzu', 2024), (49, 'Conclave', 2024), (74, "Dane-ye anjir-e ma'abed", 2024), (62, 'Dune: Part Two', 2024), (77, 'Elton John: Never Too Late', 2024), (54, 'EMILIA PÉREZ', 2024), (65, 'Gladiator II', 2024), (71, 'I Am Ready, Warden', 2024), (56, "I'm Still Here", 2024), (58, 'Inside Out 2', 2024), (82, 'Kingdom of the Planet of the Apes', 2024), (36, 'Letter to a Pig', 2024), (63, 'Maria', 2024), (59, 'Memoir of a Snail', 2024), (26, 'Nai Nai & Wài Pó', 2024), (78, 'Nickel Boys', 2024), (67, 'No Other Land', 2024), (64, 'Nosferatu', 2024), (38, 'Pachyderm', 2024), (68, 'Porcelain War', 2024), (83, 'September 5', 2024), (48, 'Sing Sing', 2024), (69, "Soundtrack to a Coup d'État", 2024), (57, 'Straume', 2024), (70, 'Sugarcane', 2024), (39, 'The A